### ЗАДАЧА: Пакетная загрузка отгрузок (try/except + custom exceptions)

Из внешней логистической системы приходят строки с отгрузками.
Нужно безопасно распарсить данные, отделить валидные записи от ошибок
и посчитать несколько итоговых метрик.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `ShipmentError`
   - `RowFormatError`
   - `WeightError`
   - `PriorityError`
   - `RegionError`.

2. Функцию `parse_shipment(row)`:
   - формат строки: `shipment_id,client,weight,priority,region`
   - `weight` должен быть числом и `> 0`
   - допустимые приоритеты: `standard`, `express`, `vip`
   - допустимые регионы: `RU`, `KZ`, `BY`
   - при ошибке конвертации веса использовать `raise ... from ...`.

3. Функцию `load_shipments(rows)`:
   - вернуть `(shipments, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных отгрузок,
   - ошибки по типам,
   - суммарный вес только для `express` и `vip`,
   - клиента-лидера по суммарному весу среди валидных записей.

In [3]:
rows = [
    'S-100,Acme,12.5,express,RU',
    'S-101,Beta,0,standard,RU',
    'S-102,Acme,abc,vip,KZ',
    'S-103,Delta,8.5,urgent,BY',
    'S-104,Gamma,15,vip,UZ',
    'S-105,Acme,4.0,standard,KZ',
    'S-106,Beta,9.5,express,BY',
]


class ShipmentError(Exception):
    pass


class RowFormatError(ShipmentError):
    pass


class WeightError(ShipmentError):
    pass


class PriorityError(ShipmentError):
    pass


class RegionError(ShipmentError):
    pass


def parse_shipment(row):
    # TODO: распарсить строку и провалидировать weight, priority, region
    # TODO: при ошибке конвертации weight использовать raise ... from ...
    arr = row.split(',')
    shipment_id, client, weight, priority, region = arr[0], arr[1], arr[2], arr[3], arr[4]
    try:
        weight = float(weight)
    except ValueError as e:
        raise WeightError('вес должен быть числом!') from e
    if priority not in ['standard', 'express', 'vip']:
        raise PriorityError('недопустимый приоритет!')
    if region not in ['RU', 'KZ', 'BY']:
        raise RegionError('некорректный регион!')
    return {'shipment_id':shipment_id, 'client': client, 'weight': weight, 'priority': priority, 'region': region}
def load_shipments(rows):
    # TODO: вернуть (shipments, errors)
    shipments = []
    errors = []
    for el in rows:
        try:
            shipment = parse_shipment(el)
            shipments.append(shipment)
        except Exception as e:
            errors.append((el, type(e).__name__, e))
    return (shipments, errors)


# TODO: вызвать load_shipments(rows)
# TODO: вывести число валидных отгрузок и число ошибок
# TODO: вывести ошибки по типам
# TODO: посчитать premium_weight только для express/vip
# TODO: найти клиента-лидера по суммарному весу
res = load_shipments(rows)
print(f'Число валидных отгрузок: {len(res[0])}')
print(f'Число ошибок: {len(res[1])}')
types_err = {}
for er in res[1]:
    types_err.setdefault(er[1], [])
    types_err[er[1]].append(er)
print(types_err)
premium_weight = 0
users_weight = {}
for el in res[0]:
    if el['priority'] == 'express' or el['priority'] == 'vip':
        premium_weight += el['weight']
    users_weight.setdefault(el['client'], 0)
    users_weight[el['client']] += el['weight']

m = 0
weight_lider = None
for key, val in users_weight.items():
    if val > m:
        m = val
        weight_lider = key
print(f'Premium weight: {premium_weight}')
print(f'лидер по весу: {weight_lider}')






Число валидных отгрузок: 4
Число ошибок: 3
{'WeightError': [('S-102,Acme,abc,vip,KZ', 'WeightError', WeightError('вес должен быть числом!'))], 'PriorityError': [('S-103,Delta,8.5,urgent,BY', 'PriorityError', PriorityError('недопустимый приоритет!'))], 'RegionError': [('S-104,Gamma,15,vip,UZ', 'RegionError', RegionError('некорректный регион!'))]}
Premium weight: 22.0
лидер по весу: Acme
